In [0]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

df = spark.table("workspace.delhi_air.ml_features_6h").toPandas()
df["hour_ist"] = pd.to_datetime(df["hour_ist"])

FEATURES = ["hour_of_day", "pm25_now", "pm25_lag1", "pm25_lag3", "pm25_lag6", "pm25_lag24",
            "temperature_c", "humidity_pct", "wind_speed_kmh", "wind_direction_deg",
            "pressure_hpa", "precipitation_mm", "boundary_layer_m"]

CUTOFF = pd.Timestamp("2026-11-03 00:00:00")
train = df[df["hour_ist"] < CUTOFF - pd.Timedelta(hours=6)]
test = df[df["hour_ist"] >= CUTOFF]
print("train rows:", len(train), "test rows:", len(test))

model = HistGradientBoostingRegressor(
    loss="absolute_error", max_iter=300, learning_rate=0.05, random_state=0
)
model.fit(train[FEATURES], train["pm25_target_6h"])
pred = model.predict(test[FEATURES])

y = test["pm25_target_6h"]
base = test["pm25_now"]
print("persistence MAE:", round(mean_absolute_error(y, base), 1))
print("model MAE:      ", round(mean_absolute_error(y, pred), 1))

high = y > 150
print("test hours with target > 150:", int(high.sum()))
print("persistence MAE on those:", round(mean_absolute_error(y[high], base[high]), 1))
print("model MAE on those:      ", round(mean_absolute_error(y[high], pred[high]), 1))
print("mean error (pred - actual), all test hours:", round((pred - y).mean(), 1))
print("mean error on high hours:", round((pred[high.values] - y[high]).mean(), 1))
print("train mean target:", round(train["pm25_target_6h"].mean(), 1), "test mean target:", round(y.mean(), 1))

In [0]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

df = spark.table("workspace.delhi_air.ml_features_6h").toPandas()
df["hour_ist"] = pd.to_datetime(df["hour_ist"])

FEATURES = ["hour_of_day", "pm25_now", "pm25_lag1", "pm25_lag3", "pm25_lag6", "pm25_lag24",
            "temperature_c", "humidity_pct", "wind_speed_kmh", "wind_direction_deg",
            "pressure_hpa", "precipitation_mm", "boundary_layer_m"]

CUTOFF = pd.Timestamp("2026-11-03 00:00:00")
train = df[df["hour_ist"] < CUTOFF - pd.Timedelta(hours=6)]
test = df[df["hour_ist"] >= CUTOFF]
print("train rows:", len(train), "test rows:", len(test))

# predict the 6-hour change, then add it back to the current reading
train_change = train["pm25_target_6h"] - train["pm25_now"]
model = HistGradientBoostingRegressor(
    loss="absolute_error", max_iter=300, learning_rate=0.05, random_state=0
)
model.fit(train[FEATURES], train_change)
pred = test["pm25_now"].values + model.predict(test[FEATURES])

y = test["pm25_target_6h"]
base = test["pm25_now"]
print("persistence MAE:", round(mean_absolute_error(y, base), 1))
print("model MAE:      ", round(mean_absolute_error(y, pred), 1))

high = y > 150
print("persistence MAE on high hours:", round(mean_absolute_error(y[high], base[high]), 1))
print("model MAE on high hours:      ", round(mean_absolute_error(y[high], pred[high.values]), 1))
print("mean error all hours:", round((pred - y).mean(), 1))
print("mean error high hours:", round((pred[high.values] - y[high]).mean(), 1))

In [0]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

df = spark.table("workspace.delhi_air.ml_features_6h").toPandas()
df["hour_ist"] = pd.to_datetime(df["hour_ist"])

FEATURES = ["hour_of_day", "pm25_now", "pm25_lag1", "pm25_lag3", "pm25_lag6", "pm25_lag24",
            "temperature_c", "humidity_pct", "wind_speed_kmh", "wind_direction_deg",
            "pressure_hpa", "precipitation_mm", "boundary_layer_m"]
H = pd.Timedelta(hours=6)

t = df["hour_ist"]
in_winter = (t >= "2025-11-01") & (t < pd.Timestamp("2026-02-01") - H)
in_autumn = (t >= "2026-09-01") & (t < pd.Timestamp("2026-11-01") - H)
train = df[in_winter | in_autumn].dropna(subset=["pm25_target_6h"])
valid = df[(t >= "2026-02-01") & (t < "2026-03-01")].dropna(subset=["pm25_target_6h"])
print("train rows:", len(train), "validation rows:", len(valid))
print("train mean:", round(train["pm25_target_6h"].mean(), 1), "validation mean:", round(valid["pm25_target_6h"].mean(), 1))

model = HistGradientBoostingRegressor(loss="absolute_error", max_iter=300, learning_rate=0.05, random_state=0)
model.fit(train[FEATURES], train["pm25_target_6h"])
pred = model.predict(valid[FEATURES])

y = valid["pm25_target_6h"]
high = y > 150
persist = valid["pm25_now"]
print("persistence MAE:", round(mean_absolute_error(y, persist), 1))
print("model MAE:", round(mean_absolute_error(y, pred), 1))
print("persistence MAE on high hours:", round(mean_absolute_error(y[high], persist[high]), 1))
print("model MAE on high hours:", round(mean_absolute_error(y[high], pred[high]), 1))
print("mean error all hours:", round((pred - y).mean(), 1))
print("mean error high hours:", round((pred[high] - y[high]).mean(), 1))

In [0]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

df = spark.table("workspace.delhi_air.ml_features_6h").toPandas()
df["hour_ist"] = pd.to_datetime(df["hour_ist"])
df = df.dropna(subset=["pm25_target_6h"])

FEATURES = ["hour_of_day", "pm25_now", "pm25_lag1", "pm25_lag3", "pm25_lag6", "pm25_lag24",
            "temperature_c", "humidity_pct", "wind_speed_kmh", "wind_direction_deg",
            "pressure_hpa", "precipitation_mm", "boundary_layer_m"]
H = pd.Timedelta(hours=6)
t = df["hour_ist"]

train = df[t < pd.Timestamp("2026-11-01") - H]
test = df[t >= "2026-11-01"]
print("train rows:", len(train), "test rows:", len(test))
print("train mean:", round(train["pm25_target_6h"].mean(), 1), "test mean:", round(test["pm25_target_6h"].mean(), 1))

model = HistGradientBoostingRegressor(loss="absolute_error", max_iter=300, learning_rate=0.05, random_state=0)
model.fit(train[FEATURES], train["pm25_target_6h"])
pred = model.predict(test[FEATURES])

y = test["pm25_target_6h"]
high = y > 150
persist = test["pm25_now"]
print("persistence MAE:", round(mean_absolute_error(y, persist), 1))
print("model MAE:", round(mean_absolute_error(y, pred), 1))
print("persistence MAE on high hours:", round(mean_absolute_error(y[high], persist[high]), 1))
print("model MAE on high hours:", round(mean_absolute_error(y[high], pred[high]), 1))
print("mean error all hours:", round((pred - y).mean(), 1))
print("mean error high hours:", round((pred[high] - y[high]).mean(), 1))

In [0]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

df = spark.table("workspace.delhi_air.ml_features_6h").toPandas()
df["hour_ist"] = pd.to_datetime(df["hour_ist"])
df = df.dropna(subset=["pm25_target_6h"])

FEATURES = ["hour_of_day", "pm25_now", "pm25_lag1", "pm25_lag3", "pm25_lag6", "pm25_lag24",
            "temperature_c", "humidity_pct", "wind_speed_kmh", "wind_direction_deg",
            "pressure_hpa", "precipitation_mm", "boundary_layer_m"]
H = pd.Timedelta(hours=6)
t = df["hour_ist"]

def evaluate(name, train, test):
    model = HistGradientBoostingRegressor(loss="absolute_error", max_iter=300, learning_rate=0.05, random_state=0)
    model.fit(train[FEATURES], train["pm25_target_6h"])
    pred = model.predict(test[FEATURES])
    y = test["pm25_target_6h"]
    persist = test["pm25_now"]
    high = y > 150
    p_mae = mean_absolute_error(y, persist)
    m_mae = mean_absolute_error(y, pred)
    print(f"--- {name} ---")
    print("train rows:", len(train), "test rows:", len(test))
    print("train mean:", round(train["pm25_target_6h"].mean(), 1), "test mean:", round(y.mean(), 1))
    print("persistence MAE:", round(p_mae, 1), "| model MAE:", round(m_mae, 1), "| improvement %:", round(100 * (p_mae - m_mae) / p_mae, 1))
    print("high hours persistence MAE:", round(mean_absolute_error(y[high], persist[high]), 1),
          "| model MAE:", round(mean_absolute_error(y[high], pred[high]), 1))
    print("mean error all hours:", round((pred - y).mean(), 1), "| high hours:", round((pred[high] - y[high]).mean(), 1))

in_winter = (t >= "2025-11-01") & (t < pd.Timestamp("2026-02-01") - H)
in_autumn = (t >= "2026-09-01") & (t < pd.Timestamp("2026-11-01") - H)
evaluate("Validation (Feb 2026)", df[in_winter | in_autumn], df[(t >= "2026-02-01") & (t < "2026-03-01")])
evaluate("Held-out check (Nov 1 onward)", df[t < pd.Timestamp("2026-11-01") - H], df[t >= "2026-11-01"])